In [1]:
###################
###   ИМПОРТЫ   ###
###################
# — импортируем вообще всё, что есть в RevPy

# самый главный декоратор
from revpy import reversible

# для объединения двух обратных функций в одну обратимую
from revpy import ReversibleFunction 

# обработка ошибок
from revpy import reversibility_assert, ReversibilityError, RevpyCompilationError

# заглушки для синтаксических конструкций (не обязательно импортировать)
from revpy import elihw, fi



#####################################
###   БАЗОВЫЕ ОБРАТИМЫЕ ФУНКЦИИ   ###
#####################################


# базовая обратимая функция: 
#   - прямая функция: добавление константы a
#   - обратная функция: вычитание константы a
add = ReversibleFunction(
    f=lambda x, a: x + a, 
    f_rev=lambda x, a: x - a
)
assert add(a=2)[3] == 5
assert add(a=2).rev[5] == 3


# базовая обратимая функция: identity
#   - название move взято от move-семантики из C++
#   - нужно для packing / unpacking
move = ReversibleFunction(
    f=lambda x: x,
    f_rev=lambda x: x,
)


# базовая обратимая функция: 
#   - прямая функция: инициализация переменной константным значением
#   - обратная функция: уничтожение этой переменной с утверждением о том, что она равна этой константе
def init(none, value, compare=lambda x, y: x == y):
    reversibility_assert(none is None, f"init: input is not None [input={none}]")
    return value

def destroy(x, value, compare=lambda x, y: x == y):
    reversibility_assert(compare(x, value), f"destroy x: x != value [x={x}, value={value}]")
    return None

init = ReversibleFunction(init, destroy)
destroy = init.rev

assert init(value=2)[None] == 2
assert destroy(value=2)[2] == None


#############################################
###   ЧИСЛА ФИБОНАЧЧИ ЧЕРЕЗ @reversible   ###
#############################################

@reversible
def fib(n):
    a, b = init((1, 1))[None]  # a, b = 1, 1
    while n > 0:
        a = add(b)[a]      # a += b
        a, b = move[b, a]  # a, b = b, a
        n = add(-1)[n]     # n -= 1 
    elihw(a >= 1 and b > 1)  
    destroy(0)[n]       # assert n == 0
    res = move[a, b]    # res = a, b
    return res


# прямой вызов:
#   * n --> fib(n), fib(n+1)
assert fib[0] == (1, 1)
assert fib[1] == (1, 2)
assert fib[2] == (2, 3)
assert fib[3] == (3, 5)
assert fib[4] == (5, 8)
assert fib[5] == (8, 13)
assert fib[6] == (13, 21)
assert fib[7] == (21, 34)

# обратный вызов:
#   * fib(n), fib(n+1) --> n
#   * если пара чисел не является 
assert fib.rev[1, 1] == 0
assert fib.rev[1, 2] == 1
assert fib.rev[2, 3] == 2
assert fib.rev[3, 5] == 3
assert fib.rev[5, 8] == 4
assert fib.rev[8, 13] == 5
assert fib.rev[13, 21] == 6
assert fib.rev[21, 34] == 7

# Если пара чисел не является последовательными числами Фибоначчи, 
# то обратный вызов завершится с ошибкой!
try: 
    fib.rev[3, 4]
except ReversibilityError as e:
    print(f"REVERSIBILITY ERROR: '{e}'")


REVERSIBILITY ERROR: 'destroy x: x != value [x=(2, 1), value=(1, 1)]'
